## 1) Setup

In [21]:

# If needed, install once:
# %pip install --upgrade sentence-transformers numpy pandas tqdm
# For local LLM via Ollama:
# %pip install --upgrade requests
# Vector store
# %pip install --upgrade chromadb

### PreDev Setup

In [22]:
from importlib import reload  # Reload modules during development
import os  # OS utilities
import requests  # HTTP requests
import numpy as np  # Numerical operations
import faiss  # Vector similarity search

import database  # Local database module
from database import AmberChromaAPI  # Amber-Chroma interface

from pypdf import PdfReader  # PDF reading
from sentence_transformers import SentenceTransformer  # Text embeddings

reload(database)  # Refresh module changes


<module 'database' from '/home/bsauce11/RAG_Prototype/Code_Saucedo/My_PreDev/database.py'>

In [23]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

In [24]:
# ## save sample documents to files
# import tempfile
# temp_dir=tempfile.mkdtemp()
#
# for i,doc in enumerate(sample_docs):
#     with open(f"{temp_dir}/doc_{i}.txt","w") as f:
#         f.write(doc)
#
# print(f"Sample document created in : {temp_dir}")

In [25]:
# ## save sample documents to files
# import tempfile
# temp_dir=tempfile.mkdtemp()
#
# for i,doc in enumerate(sample_docs):
#     with open(f"doc_{i}.txt","w") as f:
#         f.write(doc)
#
# temp_dir

### 2. Document Loading

In [26]:
# from langchain_community.document_loaders import DirectoryLoader,TextLoader
#
# # Load documents from directory
# loader = DirectoryLoader(
#     "data",
#     glob="*.txt",
#     loader_cls=TextLoader,
#     loader_kwargs={'encoding': 'utf-8'}
# )
# documents = loader.load()
#
# print(f"Loaded {len(documents)} documents")
# print(f"\nFirst document preview:")
# print(documents[0].page_content[:200] + "...")


In [27]:
# documents

### Document Splitting

In [28]:
# # Initialize text splitter
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,  # Maximum size of each chunk
#     chunk_overlap=50,  # Overlap between chunks to maintain context
#     length_function=len,
#     separators=[" "]  # Hierarchy of separators
# )
# chunks=text_splitter.split_documents(documents)
#
# print(f"Created {len(chunks)} chunks from {len(documents)} documents")
# print(f"\nChunk example:")
# print(f"Content: {chunks[0].page_content[:150]}...")
# print(f"Metadata: {chunks[0].metadata}")

In [29]:
# chunks

### Embedding Models

In [30]:
# sample_text="Machine Learning is fascinating"

In [31]:
### Huggingface model

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [32]:
# vector=embeddings.embed_query(sample_text)
# vector

### Intilialize the ChromaDB Vector Store And store the chunks in Vector Representation

In [33]:
# chunks

### Amber ChromaDB

In [34]:
# Chroma DB instance
API_CHROMA_DB = AmberChromaAPI(db_path="/opt/chromadb/data/prompt_db")
# Embedding model
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
# PDF file path
PDF_ADDRESS = "Amber25.pdf"
# Local Ollama server URL
OLLAMA_URL = "http://127.0.0.1:11434"


Using local ChromaDB path: /opt/chromadb/data/prompt_db


In [35]:
threshold_ChromaDB = 0.35 # Similarity threshold for Mails
threshold_PDF = 0.45  # Similarity threshold for PDF results

### Chunking

In [36]:
# Retrieve relevant chunks from hybrid retriever
chunks = retrieve_with_pdf(
    question,
    k_chroma=50,     # Number of ChromaDB results
    k_pdf=5,         # Number of PDF results
    threshold=threshold_ChromaDB    # Similarity threshold
)

NameError: name 'retrieve_with_pdf' is not defined

In [ ]:
# ## Create a Chromdb vector store
# persist_directory="/opt/chromadb/data/prompt_db"
#
# ## Initialize Chromadb with HuggingFace embeddings
# vectorstore=Chroma.from_documents(
#     documents=chunks,
#     embedding=HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"),
#     persist_directory=persist_directory,
#     collection_name="rag_collection"
#
# )
#
# print(f"Vector store created with {vectorstore._collection.count()} vectors")
# print(f"Vector store name: {vectorstore._collection.name}")
# print(f"Persisted to: {persist_directory}")

### Test Similarity Search

In [ ]:
# query="What are the types of machine learning?"
#
# similar_docs=vectorstore.similarity_search(query,k=3)
# similar_docs

In [ ]:
# query="what is NLP?"
#
# similar_docs=vectorstore.similarity_search(query,k=3)
# similar_docs

In [ ]:
# query="what is Deep Learning?"
#
# similar_docs=vectorstore.similarity_search(query,k=3)
# similar_docs

In [ ]:
# print(f"Query: {query}")
# print(f"\nTop {len(similar_docs)} similar chunks:")
# for i, doc in enumerate(similar_docs):
#     print(f"\n--- Chunk {i+1} ---")
#     print(doc.page_content[:200] + "...")
#     print(f"Source: {doc.metadata.get('source', 'Unknown')}")

### Advanced Similarity Search With Scores

In [ ]:
# results_scores=vectorstore.similarity_search_with_score(query,k=3)
# results_scores

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)
llm

In [ ]:
# from langchain_community.llms import Ollama
#
# #OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
# #OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
#
# llm = Ollama(model="llama3.1:8b")

In [ ]:
# Use the LLM to generate prompt
test_response=llm.invoke("What are Large Language Models")
test_response

In [ ]:
llm.invoke("What is AI")

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [ ]:
# ## Convert vector store to retriever
# retriever=vectorstore.as_retriever(
#      search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
#  )
# retriever

In [ ]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""You are a concise, technical assistant for Amber molecular simulation users. "
    Answer the user's question using ONLY the provided context.
    If the answer cannot be determined from the context, say you do not know.
    Cite sources using [Title#chunkN] notation.
    Do not speculate or introduce external knowledge.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

In [ ]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
 )

retriever

In [ ]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

In [ ]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is Deep Learning")


In [ ]:
#retriever.get_relevant_documents("What is Deep Learning")
retDocs = retriever.invoke("What is Deep Learning")
retDocs

In [ ]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    #docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [ ]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

In [ ]:
query_rag_lcel("What is machine learning?")

### Add New Documents To Existing Vector Store

In [ ]:
vectorstore

In [ ]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
in RL include: states, actions, rewards, policies, and value functions. Popular RL
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [ ]:
new_document

In [ ]:
chunks

In [ ]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [ ]:
new_doc

In [ ]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

In [ ]:
# Number of vectors in Vectore Store collection
print(f"Total vectors now: {vectorstore._collection.count()}")

In [ ]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)



In [ ]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

In [ ]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [ ]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [ ]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

In [ ]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
print("Conversational RAG chain created!")

In [ ]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [ ]:
chat_history

In [ ]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

In [ ]:
result2['answer']